# 16 · Bagging

Compare bootstrap aggregation with fully grown trees against the tuned decision tree from notebook 14. All features remain available at each split. Frozen train and validation tabular rows only.


In [1]:
from pathlib import Path
import sys,json,time
import joblib,numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd();sys.path.insert(0,str(ROOT/'src'))
from dronedet.config import load_project_config
from dronedet.features import load_tabular
from dronedet.metrics import classification_metrics,select_f1_threshold,miss_rate_at_false_alarm_rate
cfg=load_project_config(ROOT/'config.yaml');P=cfg['P']


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Inputs and baseline

Use 300 trees, bootstrap sampling, all features, and balanced class weights on the tree estimator. Baseline: fully grown tree with `max_samples=1.0`.


In [2]:
tr=load_tabular(P,'train');va=load_tabular(P,'val');Xtr=np.asarray(tr['X'],dtype=np.float32);ytr=np.asarray(tr['y'],dtype=np.int8);Xva=np.asarray(va['X'],dtype=np.float32);yva=np.asarray(va['y'],dtype=np.int8);names=np.asarray(tr['feature_names']).astype(str)
assert Xtr.shape==(7607,84) and Xva.shape==(1510,84)
assert np.array_equal(tr['clip_id'],pd.read_csv(P['splits']/'train.csv',keep_default_na=False).clip_id) and np.array_equal(va['clip_id'],pd.read_csv(P['splits']/'val.csv',keep_default_na=False).clip_id)
seeds=[int(cfg['seed']),int(cfg['seed'])+1,int(cfg['seed'])+2]
tree14=joblib.load(ROOT/'results'/'models'/'14_decision_tree.joblib')['model']
tuned=DecisionTreeClassifier(criterion='gini',max_depth=tree14.tree_.max_depth,min_samples_leaf=1,ccp_alpha=float(tree14.ccp_alpha),class_weight='balanced')
default={'max_samples':1.0,'base':'full'}
configs=[{'max_samples':m,'base':b} for m in (.5,.8,1.0) for b in ('full','tuned')]
def estimator(base):return DecisionTreeClassifier(class_weight='balanced') if base=='full' else __import__('sklearn').base.clone(tuned)
def fit_eval(hp,seed,n=300):
 m=BaggingClassifier(estimator=estimator(hp['base']),n_estimators=n,max_samples=hp['max_samples'],max_features=1.0,bootstrap=True,n_jobs=-1,random_state=seed);m.fit(Xtr,ytr);sc=m.predict_proba(Xva)[:,1];th=select_f1_threshold(yva,sc);met=classification_metrics(yva,sc,threshold=th);return m,{'threshold':float(th),'val_f1':met['f1'],'val_roc_auc':met['roc_auc'],'far':met['false_alarm_rate'],'miss':met['miss_rate']}
rows=[];models={};t0=time.perf_counter()
for ci,hp in enumerate(configs):
 for seed in seeds:
  m,met=fit_eval(hp,seed);rows.append({'config_id':ci,'seed':seed,**hp,**met});models[(ci,seed)]=m
frame=pd.DataFrame(rows);base=float(frame[frame.config_id==4].val_f1.mean());sd=float(frame[frame.config_id==4].val_f1.std())
sumry=frame.groupby('config_id',as_index=False).agg(mean_f1=('val_f1','mean'),std_f1=('val_f1','std'),mean_auc=('val_roc_auc','mean'));sumry['delta']=sumry.mean_f1-base;sumry['accepted']=(sumry.config_id!=4)&(sumry.delta>=.01)&(sumry.delta>sd)
accepted=sumry[sumry.accepted].config_id.tolist();combo_id=None
if len(accepted)>=2:
 combo={'max_samples':1.0,'base':'full'}
 for ci in accepted: combo.update({k:v for k,v in configs[ci].items() if v!=default.get(k)})
 combo_id=len(configs);configs.append(combo)
 for seed in seeds:
  m,met=fit_eval(combo,seed);frame.loc[len(frame)]={'config_id':combo_id,'seed':seed,**combo,**met};models[(combo_id,seed)]=m
 cm=frame[frame.config_id==combo_id].val_f1.mean();best_single=sumry[sumry.config_id.isin(accepted)].mean_f1.max()
 if cm<best_single+.01:combo_id=None
valid=[4]+[int(i) for i in accepted if i!=combo_id]+([combo_id] if combo_id is not None else [])
summary=frame[frame.config_id.isin(valid)].groupby('config_id',as_index=False).agg(mean_f1=('val_f1','mean'),std_f1=('val_f1','std'),mean_auc=('val_roc_auc','mean'))
chosen_id=int(summary.sort_values(['mean_f1','config_id'],ascending=[False,True]).iloc[0].config_id);best=frame[frame.config_id==chosen_id].sort_values(['val_f1','seed'],ascending=[False,True]).iloc[0];seed=int(best.seed);model=models[(chosen_id,seed)]
score=model.predict_proba(Xva)[:,1];threshold=select_f1_threshold(yva,score);metrics=classification_metrics(yva,score,threshold=threshold)
print('18 fits; accepted:',accepted,'combined retained:',combo_id is not None,'selected:',configs[chosen_id],metrics['f1'])


18 fits; accepted: [] combined retained: False selected: {'max_samples': 1.0, 'base': 'full'} 0.8342245989304813


## Saturation diagnostic and freeze

The warm-start curve trains the default to 50, 100, 200 and 300 trees. This is diagnostic; it does not alter the 300-tree selection.


In [3]:
warm=BaggingClassifier(estimator=estimator('full'),n_estimators=50,max_samples=1.0,max_features=1.0,bootstrap=True,n_jobs=-1,random_state=seeds[0],warm_start=True);curve=[]
for n in (50,100,200,300):
 warm.set_params(n_estimators=n);warm.fit(Xtr,ytr);curve.append({'trees':n,'val_auc':float(__import__('sklearn').metrics.roc_auc_score(yva,warm.predict_proba(Xva)[:,1]))})
miss1,thr1=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.01);miss5,thr5=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.05)
FIG=ROOT/'results'/'figures';TABLES=ROOT/'results'/'tables';MODELS=ROOT/'results'/'models';CARDS=ROOT/'results'/'cards'
for d in (FIG,TABLES,MODELS,CARDS):d.mkdir(parents=True,exist_ok=True)
frame.to_csv(TABLES/'16_bagging_seed_metrics.csv',index=False);pd.DataFrame(curve).to_csv(TABLES/'16_bagging_saturation.csv',index=False)
fig,ax=plt.subplots(figsize=(7,5));ax.plot([x['trees'] for x in curve],[x['val_auc'] for x in curve],'o-');ax.set(xlabel='Trees',ylabel='Validation ROC-AUC',title='Bagging saturation');fig.tight_layout();fig.savefig(FIG/'16_bagging_saturation.png',dpi=160);plt.close(fig)
card={'model':'bagging','representation':'84 tabular features','best_hyperparameters':configs[chosen_id],'val_metrics':{**metrics,'threshold':float(threshold),'miss_rate_at_far_1pct':float(miss1),'threshold_at_far_1pct':float(thr1),'miss_rate_at_far_5pct':float(miss5),'threshold_at_far_5pct':float(thr5)},'seeds':seeds,'selected_seed':seed,'train_time_s':float(time.perf_counter()-t0),'diagnostics':{'trees':300,'accepted_variants':accepted,'combination_retained':combo_id is not None},'decision_log_ids':['D044','D045']}
fh=json.loads((P['spec']/'frontend_hash.json').read_text(encoding='utf-8'))['params_hash'];joblib.dump({'model':model,'feature_names':names.tolist(),'threshold':float(threshold),'threshold_rule':'validation_f1_max_ties_higher','front_end_hash':fh,'hyperparameters':card['best_hyperparameters']},MODELS/'16_bagging.joblib');(CARDS/'16_bagging.json').write_text(json.dumps(card,indent=2,default=float),encoding='utf-8');print(metrics)


{'accuracy': 0.9589403973509933, 'precision': 0.8253968253968254, 'recall': 0.8432432432432433, 'f1': 0.8342245989304813, 'roc_auc': 0.9674023457419685, 'false_alarm_rate': 0.02490566037735849, 'miss_rate': 0.15675675675675677, 'confusion_matrix': [[1292, 33], [29, 156]]}
